**实验目标：**

通过本实验，你将深入了解和实践说话人识别技术，并掌握利用声音特征进行有效说话人识别的基本方法，了解不同特征和模型对识别准确率的影响。

实验的核心目标是使用TIMIT数据集来训练一个说话人识别系统，涵盖数据预处理、特征提取、模型训练和评估等关键步骤。


**实验方法：**

**1. 数据预处理和划分(可选)：**
  - 数据集下载：
    - 大陆访问url: https://yun.139.com/shareweb/#/w/i/2ur4mcv5Y4qc7  提取码:yn8x（4月27日18点前可访问）
    - 谷歌访问url: https://drive.google.com/file/d/180mSIiXN9RVDV2Xn1xcWNkMRm5J5MjN4/view?usp=sharing
  - 为了方便大家，我们提供了划分好的TIMIT数据集结构，当然你也可以根据需求自行划分该数据集。
  - 为简化难度，我们排除了SA的两个方言句子，并在剩余的8个句子中选取了SX的5个句子和SI的1个句子作为训练集，SI的另外2个句子作为测试集。
  - 该链接下载的数据集只保留了音频文件，完整数据集（包含音频对应文本、标注等信息）可参见备注链接下载。
  
**2. 特征提取：**
  - 学习并实现包括但不限于MFCC特征等特征的提取，探索声音信号的频率和时间特性。
  - 鼓励尝试和比较其他特征提取方法，例如LPCC或声谱图特征，以理解不同特征对识别性能的影响。
  
**3. 模型选择和训练：**
  - 探索并选择适合的分类器和模型进行说话人识别，如GMM、Softmax分类器或深度学习模型。
  - 实现模型训练流程，使用训练集数据训练模型。
  
**4. 评估和分析：**
  - 使用准确率作为主要的评价指标在测试集上评估模型性能。
  - 对比不同特征和模型的性能，分析其对说话人识别准确率的影响。
  - 可视化不同模型的识别结果和错误率，讨论可能的改进方法。

**实验要求：**
  - 1.选择并实现至少一种特征的提取，并鼓励尝试其他特征提取方法。
  - 2.选择并实现至少一种分类器或模型进行说话人识别，并使用准确率评估指标评估其性能。
  - 3.通过实验对比、分析和可视化，撰写详细的实验报告，包括实验目的、实验方法、结果分析和结论。
  - 4.实验报告应以清晰、逻辑性强的形式呈现，图表和结果应清楚明了。

**其他说明：**
  - 实验的最终打分环节会看识别性能，会对原理和实现代码部分做抽查提问，综合评定成绩。
  - 我们**鼓励做原创性探索**，即使性能不是很好，但有创新性、有价值、有意义的探索和尝试会有额外加分。

## 1. 实验准备

In [ ]:
## 导入必要的库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

import os
import json
import numpy as np
import librosa
import warnings

import seaborn as sns
from sklearn.metrics import confusion_matrix
import librosa.display
import matplotlib.pyplot as plt

# 可以根据需要导入其他库，比如librosa用于音频处理

: 

## 2. 数据预处理(加载数据集)

In [ ]:
# 数据集基本信息如下
# 方言地区：DR1～DR8
# 性别：F/M
# 说话者ID：3个大写字母+1个阿拉伯数字
# 句子ID：句子类型（SA/SI/SX）+编号
# 详细介绍参见 https://blog.csdn.net/qq_39373179/article/details/103788208

# 加载数据和特征提取相关函数

# 路径请根据实际情况修改
TrainJson = "test_info.json"  # 训练集
TestJson = "train_info.json"  # 测试集
BaseDatasetDir = "DATA/TRAIN"  # 音频数据根目录

warnings.filterwarnings('ignore')

"""提取音频的 MFCC 特征"""
def extract_features(file_path, n_mfcc=20):
    y, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfcc.T


"""
1.读取 json 文件
通过 load_data_from_json 函数，
读取 test_info.json（训练集）和 train_info.json（测试集），
获取每个音频样本的文件路径和说话人标签。

2.定位音频文件
根据 json 文件中的 filepath 字段和 BaseDatasetDir 路径，
拼接出每个音频的实际存储路径。

3.提取音频特征
对每个音频文件，调用 extract_features 函数，
使用 librosa 库提取 MFCC 特征（梅尔频率倒谱系数），
将原始音频信号转化为适合机器学习的特征矩阵。

4.生成特征和标签列表
将所有音频的 MFCC 特征存入 train_features/test_features，
将对应的说话人标签存入 train_labels/test_labels，
供后续模型训练和评估使用。
"""

def load_data_from_json(json_path, base_dir):
    features = []
    labels = []

    if not os.path.exists(json_path):
        print(f"[!] 致命错误：找不到文件 {json_path}")
        return features, labels

    print(f"[*] 正在努力解析 {json_path} ...")

    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()

        if content.startswith('{') and content.endswith('}'):
            content = f"[{content}]"

        try:
            data_info = json.loads(content)
        except json.JSONDecodeError:
            data_info = ast.literal_eval(content)

        success_count = 0
        for item in data_info:
            rel_path = item.get("filepath", item.get("file_path", item.get("file")))
            speaker_id = item.get("speaker_id", item.get("speaker", item.get("label")))

            if not rel_path or not speaker_id:
                continue

            # 统一斜杠，避免出现类似 TIMIT/TRAIN\DR1 这样的混合斜杠
            rel_path = rel_path.replace('\\', '/').strip('/')
            full_path = os.path.join(base_dir, rel_path)
            full_path = os.path.normpath(full_path)  # 标准化路径

            # --- 核心：自动找文件逻辑（解决后缀大小写和下划线问题） ---
            possible_paths = [
                full_path,  # 原路径
                full_path.replace('.wav', '.WAV'),  # 尝试大写 WAV后缀
                full_path.replace('_.wav', '.WAV'),  # 尝试去掉下划线的大写后缀
                full_path.replace('_.wav', '.wav')  # 尝试去掉下划线的小写后缀
            ]

            actual_path = None
            for p in possible_paths:
                if os.path.exists(p):
                    actual_path = p
                    break

            if actual_path:
                mfcc_feat = extract_features(actual_path)
                features.append(mfcc_feat)
                labels.append(speaker_id)
                success_count += 1
            else:
                # 打印绝对路径，方便你排查到底去哪里找了
                print(f"   [!] 找不到音频，已尝试查找路径: {os.path.abspath(full_path)}")

        if success_count == 0:
            print(f"\n[!] 警告：没有找到任何有效的音频文件！")
            print(f"请检查 BaseDatasetDir 的值 (目前是 '{base_dir}') 是否确实包含了 DR1 等文件夹。")
        else:
            print(f"[*] 成功加载 {success_count} 条音频数据！\n")

    except Exception as e:
        print(f"[!] 解析彻底失败！请检查文件内容。错误信息：{e}")

    return features, labels

train_features, train_labels = load_data_from_json(TrainJson, BaseDatasetDir)
test_features, test_labels = load_data_from_json(TestJson, BaseDatasetDir)

## 3. 特征提取

In [ ]:
# 在2预处理完成后，train_features 和 test_features 分别包含了训练集和测试集的 MFCC 特征，
# 输出一些基本信息，验证特征提取是否成功
# 训练集MFCC特征提取示例
print(f"训练集样本数: {len(train_features)}")
if train_features:
    print(f"训练集第一个样本的MFCC特征形状: {train_features[0].shape}")
else:
    print("训练集无数据")

# 测试集MFCC特征提取示例
print(f"测试集样本数: {len(test_features)}")
if test_features:
    print(f"测试集第一个样本的MFCC特征形状: {test_features[0].shape}")
else:
    print("测试集无数据")

## 4. 模型选择和训练

In [ ]:
# 以GMM为例进行说话人识别模型训练
from sklearn.mixture import GaussianMixture
speaker_models = {}
if len(train_labels) > 0:
    unique_speakers = np.unique(train_labels)
    gmm_n_components = 16  # 高斯分量数
    for speaker in unique_speakers:
        speaker_feats = [train_features[i] for i, label in enumerate(train_labels) if label == speaker]
        X_train = np.vstack(speaker_feats)
        gmm = GaussianMixture(n_components=gmm_n_components, covariance_type='diag', max_iter=200, random_state=42)
        gmm.fit(X_train)
        speaker_models[speaker] = gmm
    print(f"已为{len(unique_speakers)}个说话人训练GMM模型。")
else:
    print("训练数据为空，无法训练模型。")

## 5. 评价指标(准确率Accuracy)

In [ ]:
# 在测试集上预测并计算准确率
predicted_labels = []
if len(test_labels) > 0 and speaker_models:
    for test_feat in test_features:
        best_score = -np.inf
        best_speaker = None
        for speaker, gmm in speaker_models.items():
            score = gmm.score(test_feat)
            if score > best_score:
                best_score = score
                best_speaker = speaker
        predicted_labels.append(best_speaker)
    accuracy = accuracy_score(test_labels, predicted_labels)
    print(f"测试集准确率: {accuracy * 100:.2f}%")
else:
    print("测试数据为空或模型未训练。")

##  6. 分析和可视化

In [ ]:
# 可视化混淆矩阵和MFCC特征


if len(test_labels) > 0 and predicted_labels:
    unique_speakers = np.unique(test_labels)
    cm = confusion_matrix(test_labels, predicted_labels, labels=unique_speakers)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=unique_speakers, yticklabels=unique_speakers)
    plt.title('说话人识别混淆矩阵 (GMM-MFCC)', fontsize=16)
    plt.xlabel('预测标签 (Predicted Label)', fontsize=12)
    plt.ylabel('真实标签 (True Label)', fontsize=12)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.show()

    # 展示一个测试样本的MFCC特征图谱
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(test_features[0].T, x_axis='time', sr=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'说话人 {test_labels[0]} 的测试音频 MFCC 特征图谱')
    plt.tight_layout()
    plt.show()

## 7. 结果讨论
讨论你的模型性能，尝试解释为什么某些模型比其他模型表现好，以及可能的改进方法。

## 8. 保存模型（可选）
如果需要，可以在这里添加代码保存你的模型。